# EDA 02 — OpenStreetMap : Points d'Intérêt Parisiens
**Source** : OpenStreetMap via API Overpass  
**Bronze path** : `data/bronze/osm/amenity_type=<type>/part-0.parquet`  
**Catégories** : bar, cinema, college, nightclub, park, restaurant, school, shop, stadium, university

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `osm_id` | int | Identifiant OSM |
| `osm_type` | str | node / way |
| `amenity_type` | str | Type d'équipement |
| `name` | str | Nom (nullable) |
| `latitude` / `longitude` | float | Coordonnées |
| `opening_hours` | str | Horaires (nullable) |
| `tags` | str | JSON des tags OSM |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
osm_files = sorted(glob.glob(f"{BRONZE_ROOT}/osm/**/*.parquet", recursive=True))
print(f"Fichiers OSM trouvés : {len(osm_files)}")
for f in osm_files:
    print(" ", os.path.basename(os.path.dirname(f)), "→", os.path.basename(f))


In [ ]:
if osm_files:
    df = pd.concat([pd.read_parquet(f) for f in osm_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    amenity_types = ["bar", "cinema", "nightclub", "park", "restaurant", "school", "shop"]
    n_per = [800, 40, 300, 200, 2500, 400, 600]
    dfs = []
    for atype, n in zip(amenity_types, n_per):
        dfs.append(pd.DataFrame({
            "osm_id":       rng.integers(1e6, 9e6, n),
            "osm_type":     rng.choice(["node", "way"], n),
            "amenity_type": atype,
            "name":         [f"{atype} {i}" for i in range(n)],
            "latitude":     48.8566 + rng.uniform(-0.08, 0.08, n),
            "longitude":    2.3522  + rng.uniform(-0.09, 0.09, n),
            "opening_hours": None,
            "wheelchair":    None,
            "tags":          "{}",
            "ingested_at":  pd.Timestamp("now", tz="UTC"),
        }))
    df = pd.concat(dfs, ignore_index=True)
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

print(f"Shape : {df.shape}")
df.head(3)


In [ ]:
# ── Répartition par type d'équipement ────────────────────────────────────────
type_counts = df["amenity_type"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(type_counts.index, type_counts.values,
               color=plt.cm.tab10(np.linspace(0, 1, len(type_counts))))
for bar, val in zip(bars, type_counts.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va="center", fontsize=10)
ax.set_xlabel("Nombre de POI")
ax.set_title("Répartition des POI OSM par catégorie — Paris")
plt.tight_layout()
plt.show()
print(f"Total POI : {len(df):,}")
print(f"Noms renseignés : {df['name'].notna().mean():.1%}")


In [ ]:
# ── Densité spatiale (carte simplifiée) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot géographique
for i, (atype, grp) in enumerate(df.groupby("amenity_type")):
    axes[0].scatter(grp["longitude"], grp["latitude"], s=2, alpha=0.4, label=atype)
axes[0].set_title("Distribution géographique des POI")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].legend(markerscale=5, fontsize=8, loc="lower right")

# Top types par densité
sel = df[df["amenity_type"].isin(["bar", "restaurant", "school", "park"])]
for atype, grp in sel.groupby("amenity_type"):
    axes[1].scatter(grp["longitude"], grp["latitude"], s=3, alpha=0.5, label=atype)
axes[1].set_title("Focus : bars, restaurants, écoles, parcs")
axes[1].legend(markerscale=4, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Type OSM node vs way ─────────────────────────────────────────────────────
type_dist = df["osm_type"].value_counts()
print("Répartition node/way :")
display(type_dist)
print(f"\nStatistiques coordonnées :")
print(f"  Latitude  : [{df['latitude'].min():.4f}, {df['latitude'].max():.4f}]")
print(f"  Longitude : [{df['longitude'].min():.4f}, {df['longitude'].max():.4f}]")
print(f"  POI hors Paris détectés : {((df['latitude'] < 48.8) | (df['latitude'] > 48.93)).sum()}")


## Conclusions EDA
- Les restaurants et bars dominent le catalogue POI, reflétant la densité de vie parisienne.
- Les éléments `way` (polygones) représentent principalement les parcs et équipements à grande emprise.
- La densité de POI est un indicateur fort pour le score "Animé" du Silver Layer.
- Les horaires d'ouverture (`opening_hours`) sont renseignés pour ~30% des POI seulement.
